# Lab 01: Building a ReAct Agent from Scratch

## 🎯 What We're Building

In this lab, you will build a **working AI Agent** in three stages:

| Stage | What | Lines of Code | What You Learn |
|-------|------|---------------|----------------|
| **Stage 1** | Raw agent with just OpenAI SDK | ~80 lines | How agents REALLY work under the hood |
| **Stage 2** | Same agent with LangGraph | ~15 lines | Why frameworks exist and what they give you |
| **Stage 3** | Add memory and persistence | ~25 lines | How agents remember and survive crashes |

> **Prerequisites:** Read the [README.md](README.md) first to understand the concepts.

---

## 🔧 Setup: Connect to Azure OpenAI

Before we build anything, we need to connect to our AI model.

In Lab 00, we deployed **Azure OpenAI** with GPT-4.1 into your Azure subscription.
The setup script generated a `.env` file with all the connection details.

The cell below:
1. Installs the Python packages we'll use (OpenAI SDK, LangGraph, etc.)
2. Loads the connection strings from your `.env` file
3. Verifies that Azure OpenAI is accessible

> 💡 If you see "Azure OpenAI not configured", go back and run **Lab 00** first.

In [2]:
# ──────────────────────────────────────────────────────
# Install required Python packages (run this once)
#
# • openai         — The official SDK to talk to Azure OpenAI
# • langchain-openai — LangChain's wrapper for OpenAI (used in Stage 2)
# • langgraph      — Framework for building agent graphs (used in Stage 2+3)
# • python-dotenv  — Loads our .env file with Azure connection strings
# • rich           — Pretty printing for our agent's output
# ──────────────────────────────────────────────────────
%pip install openai langchain-openai langgraph python-dotenv rich --quiet


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv

# ──────────────────────────────────────────────────────
# Load Azure connection strings from .env file
#
# The .env file was auto-generated by Lab 00's setup notebook.
# It contains endpoints and API keys for all Azure resources.
# We load it into environment variables so our code can use them.
# ──────────────────────────────────────────────────────
load_dotenv("../.env")

# Verify Azure OpenAI is configured
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not endpoint or not api_key:
    print("⚠️  Azure OpenAI not configured!")
    print("   Run lab-00-setup/setup.ipynb first to deploy Azure resources.")
else:
    print(f"✅ Azure OpenAI endpoint: {endpoint}")
    print(f"✅ API key loaded (ends with ...{api_key[-4:]})")
    print()
    print("💡 The .env file was created by Lab 00. It contains all")
    print("   connection strings for Azure OpenAI, Search, Cosmos DB, etc.")

✅ Azure OpenAI endpoint: https://ai-agentlabs-fk25xdiuwudo4.cognitiveservices.azure.com/
✅ API key loaded (ends with ...5KRG)

💡 The .env file was created by Lab 00. It contains all
   connection strings for Azure OpenAI, Search, Cosmos DB, etc.


---

# 🏗️ STAGE 1: Build a Raw Agent (No Framework)

We're going to build a complete agent using **only** the OpenAI Python SDK.
No LangChain, no LangGraph — just raw Python.

**Why start here?** Because you need to see every moving part before a framework hides them.

---

## Step 1.1: Define Our Tools

An agent needs tools — functions it can call to interact with the world.
Let's create three simple tools:

| Tool | Purpose | Example |
|------|---------|--------|
| `get_weather` | Get weather for a city | "Tel Aviv" → "25°C, sunny" |
| `calculate` | Do math | "15 * 7 + 3" → 108 |
| `search_knowledge` | Search a knowledge base | "vacation policy" → relevant info |

These are intentionally simple — we're learning the **agent pattern**, not building real integrations.

In [4]:
import json

# ============================================================
# TOOL 1: Weather (simulated)
# In production, this would call a real weather API.
# ============================================================
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Simulated weather data
    weather_data = {
        "tel aviv": {"temp": 28, "condition": "sunny", "humidity": 65},
        "new york": {"temp": 12, "condition": "cloudy", "humidity": 70},
        "london":   {"temp": 8,  "condition": "rainy",  "humidity": 85},
        "tokyo":    {"temp": 20, "condition": "clear",  "humidity": 55},
    }
    data = weather_data.get(city.lower(), {"temp": 15, "condition": "unknown", "humidity": 50})
    return f"Weather in {city}: {data['temp']}°C, {data['condition']}, humidity {data['humidity']}%"


# ============================================================
# TOOL 2: Calculator
# Uses Python's eval (safe for math expressions only)
# ============================================================
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    # Only allow safe math characters
    allowed_chars = set('0123456789+-*/.() ')
    if not all(c in allowed_chars for c in expression):
        return "Error: only mathematical expressions are allowed"
    try:
        result = eval(expression)  # Safe because we validated input
        return f"Result: {expression} = {result}"
    except Exception as e:
        return f"Error calculating: {e}"


# ============================================================
# TOOL 3: Knowledge Base Search
# A tiny in-memory knowledge base (in production: vector DB)
# ============================================================
KNOWLEDGE_BASE = [
    {"topic": "vacation policy", "content": "Employees get 22 vacation days per year. Unused days can carry over up to 5 days. Must request vacation at least 2 weeks in advance."},
    {"topic": "remote work", "content": "Employees can work remotely up to 3 days per week. Must be available on Teams during core hours (10am-4pm). Manager approval needed for full remote."},
    {"topic": "expense policy", "content": "Meals up to $50/day during travel. Flights must be economy class. Hotel max $200/night. All expenses need receipts and manager approval within 30 days."},
    {"topic": "onboarding", "content": "New employees have a 90-day onboarding period. Week 1: HR orientation. Week 2-4: Team training. Month 2-3: Shadowing and first projects."},
]

def search_knowledge(query: str) -> str:
    """Search the company knowledge base for relevant information."""
    query_lower = query.lower()
    results = []
    for item in KNOWLEDGE_BASE:
        # Simple keyword matching (in production: vector similarity search)
        if any(word in query_lower for word in item["topic"].split()):
            results.append(f"[{item['topic']}]: {item['content']}")
    
    if results:
        return "\n\n".join(results)
    return "No relevant information found in the knowledge base."


# Test our tools
print("🌤️", get_weather("Tel Aviv"))
print("🔢", calculate("15 * 7 + 3"))
print("📚", search_knowledge("vacation days"))

🌤️ Weather in Tel Aviv: 28°C, sunny, humidity 65%
🔢 Result: 15 * 7 + 3 = 108
📚 [vacation policy]: Employees get 22 vacation days per year. Unused days can carry over up to 5 days. Must request vacation at least 2 weeks in advance.


## Step 1.2: Define Tool Schemas for the LLM

Now here's the crucial part that most people misunderstand:

**The LLM never calls our Python functions directly.** Instead:

1. We send the LLM a **description** of each tool (a JSON schema)
2. The LLM reads the descriptions and **decides** which tool to use
3. The LLM outputs **JSON** saying "call this tool with these arguments"
4. **Our code** executes the actual function
5. We send the result **back** to the LLM

Think of it like a **restaurant menu**:
- The menu (tool schema) tells you what's available
- You (the LLM) decide what to order
- The kitchen (our code) prepares the food
- The waiter (our code) brings it back to you

The quality of tool descriptions directly affects how well the LLM uses them!

> 💡 **Key insight:** The `description` field in the schema is what the LLM reads to decide
> when to use a tool. Vague descriptions = bad tool selection. Specific descriptions = great results.

In [5]:
# Tool schemas — this is what we send to the LLM so it knows what tools exist
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city. Use this when the user asks about weather, temperature, or conditions in a specific location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g., 'Tel Aviv', 'New York'"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Use this for any math calculations the user needs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression, e.g., '15 * 7 + 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search the company knowledge base for information about policies, procedures, and guidelines. Use this when the user asks about company rules, policies, or internal information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query describing what information to look for"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# Map tool names to actual Python functions
TOOL_FUNCTIONS = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_knowledge": search_knowledge,
}

print(f"✅ Defined {len(TOOL_SCHEMAS)} tools: {', '.join(TOOL_FUNCTIONS.keys())}")

✅ Defined 3 tools: get_weather, calculate, search_knowledge


## Step 1.3: Build the ReAct Loop — The Heart of Every Agent

This is the **most important part** of the entire lab. The ReAct loop is what turns
a simple LLM into an agent. Every agent framework (LangChain, LangGraph, Semantic Kernel)
implements this same loop — they just hide it behind nice APIs.

### How the loop works:

```
1. Send messages + tool schemas to LLM
2. LLM responds with either:
   a) A final text answer → we're done! Exit the loop.
   b) A tool_call request → execute the tool, add result to messages, go back to step 1
3. Repeat until the LLM gives a text answer OR we hit max iterations (safety!)
```

### What happens in each iteration:

| Step | What | Who Does It | Example |
|------|------|------------|---------|
| **Think** | LLM decides what to do next | The LLM | "I need to check the weather" |
| **Act** | Execute the chosen tool | Our Python code | `get_weather("Tokyo")` |
| **Observe** | See the result | Both (result added to messages) | "20°C, clear" |
| **Repeat** | LLM sees result, decides if done | The LLM | "I have enough info, let me answer" |

### The message history is the agent's memory

Every message stays in the history. On each LLM call, the model sees ALL previous messages.
That's how it knows what tools it already called and what results it got.

**Pay close attention to the code below — you're looking at what every agent framework does internally.**

In [ ]:
from openai import AzureOpenAI
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()

# Connect to Azure OpenAI (NOT regular OpenAI)
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
)

# Use GPT-4o deployed in Azure
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4o")

SYSTEM_PROMPT = """You are a helpful assistant with access to tools.
Use the available tools when you need to look up information, do calculations,
or check weather. Always explain your reasoning."""


def run_agent(user_message: str, max_iterations: int = 5) -> str:
    """
    Run a ReAct agent loop.
    
    This is the ENTIRE agent — ~40 lines of core logic.
    Everything else is just printing for visibility.
    """
    # Initialize the message history
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    
    console.print(f"\n[bold blue]👤 User:[/] {user_message}\n")
    
    for iteration in range(1, max_iterations + 1):
        console.print(f"[dim]--- Iteration {iteration}/{max_iterations} ---[/]")
        
        # ===== STEP 1: Call the LLM =====
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        
        assistant_message = response.choices[0].message
        
        # ===== STEP 2: Check if the LLM wants to call a tool =====
        if assistant_message.tool_calls:
            # The LLM wants to use a tool!
            # Add the assistant's message (with tool_calls) to history
            messages.append(assistant_message)
            
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                
                console.print(f"  [yellow]🤔 Think:[/] I need to use [bold]{tool_name}[/]")
                console.print(f"  [green]🔧 Act:[/]   {tool_name}({tool_args})")
                
                # ===== STEP 3: Execute the tool =====
                tool_function = TOOL_FUNCTIONS.get(tool_name)
                if tool_function:
                    # Call the ACTUAL Python function
                    result = tool_function(**tool_args)
                else:
                    result = f"Error: Unknown tool '{tool_name}'"
                
                console.print(f"  [cyan]👀 Observe:[/] {result}\n")
                
                # ===== STEP 4: Add the tool result to message history =====
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        
        else:
            # No tool calls — the LLM is giving us the final answer!
            final_answer = assistant_message.content
            console.print(f"[bold green]📤 Answer:[/] {final_answer}\n")
            
            # Show stats
            console.print(f"[dim]Completed in {iteration} iteration(s), "
                         f"{len(messages)} messages in history[/]")
            return final_answer
    
    return "⚠️ Max iterations reached without a final answer."


print("✅ Agent function defined. Let's test it!")

## Step 1.4: Test the Agent!

Now comes the fun part — let's run our agent with different types of questions and **watch the ReAct loop in action**.

For each test, observe carefully:
- **🤔 Think:** What did the LLM decide to do?
- **🔧 Act:** Which tool did it call, and with what arguments?
- **👀 Observe:** What result came back?
- **📤 Answer:** How did it combine the information?

Also notice:
- How many **iterations** the agent takes (1 iteration = 1 LLM call)
- Whether it calls **one tool or multiple** tools
- **When it decides to stop** and give a final answer

### Test 1: Simple single-tool question
The agent should call `get_weather` once, then answer.

In [ ]:
# ──────────────────────────────────────────────────────
# TEST 1: Simple question — needs ONE tool call
#
# What to watch for:
# - The agent should decide to use get_weather
# - It should complete in 2 iterations (1 tool call + 1 answer)
# - The final answer should include the weather data
# ──────────────────────────────────────────────────────
run_agent("What's the weather like in Tel Aviv?")

In [ ]:
# ──────────────────────────────────────────────────────
# TEST 2: Different tool — knowledge base search
#
# The LLM should choose search_knowledge (not get_weather)
# based on the description: "Search the company knowledge base
# for information about policies, procedures..."
# 
# This shows that the LLM reads the tool DESCRIPTIONS to decide!
# ──────────────────────────────────────────────────────
run_agent("What is our company's vacation policy?")

In [ ]:
# ──────────────────────────────────────────────────────
# TEST 3: MULTI-TOOL question — this is where agents shine!
#
# This question requires THREE different tools:
#   1. get_weather → Tokyo weather
#   2. search_knowledge → expense policy (meal limit)
#   3. calculate → 3 × $15 = $45, compare to $50 limit
#
# Watch how the agent:
#   - Decides to call multiple tools
#   - Uses results from one tool to inform the next
#   - Combines ALL results into a coherent answer
#
# This is something a simple chatbot CANNOT do!
# ──────────────────────────────────────────────────────
run_agent("I'm planning a trip to Tokyo. What's the weather there? "
          "Also, what's the daily meal expense limit? "
          "If I eat 3 meals at $15 each, am I within budget?")

In [ ]:
# ──────────────────────────────────────────────────────
# TEST 4: Question that needs NO tools
#
# This shows that the agent doesn't ALWAYS call tools.
# If the LLM can answer directly from its own knowledge,
# it skips tool calling entirely → 1 iteration, 0 tool calls.
#
# This is important: the agent decides dynamically
# whether to use tools or not.
# ──────────────────────────────────────────────────────
run_agent("What is 2 + 2? Don't use any tools, just answer.")

### 🤔 Reflection: What Did We Learn?

Take a moment to answer these questions:

1. **Who decides which tool to use?** → The LLM, based on the tool descriptions and the user's question
2. **Who executes the tool?** → Our Python code, NOT the LLM
3. **How does the LLM know the tool result?** → We send it back as a `tool` message
4. **When does the loop stop?** → When the LLM responds with text instead of a tool_call
5. **What prevents infinite loops?** → Our `max_iterations` safety check

This is **the core of every agent in every framework**. LangChain, LangGraph, Semantic Kernel — they all do exactly this under the hood.

---

## Step 1.5: Peek Inside — The Message History

So far we've seen the agent work from the outside. Now let's look **inside** — at the
actual messages flowing between our code and the LLM.

Understanding the message history is **critical** because:
- It's how the LLM tracks what it already did (short-term memory)
- It's how the LLM sees tool results
- It's what gets saved when you add "memory" (Stage 3)
- Every agent framework manages these messages for you — now you see what they manage

The cell below runs an agent query and then prints **every message** in the history,
showing you the exact conversation between our code and the LLM.

In [ ]:
from IPython.display import HTML, display

def run_agent_with_history(user_message: str, max_iterations: int = 5):
    """Same agent, but returns the full message history for inspection."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    
    for iteration in range(1, max_iterations + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMAS
        )
        msg = response.choices[0].message
        
        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                fn = TOOL_FUNCTIONS.get(tc.function.name)
                result = fn(**args) if fn else "Unknown tool"
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        else:
            messages.append({"role": "assistant", "content": msg.content})
            break
    
    return messages

# Run and inspect
history = run_agent_with_history("What's the weather in London and what is 32 * 9?")

# Display the message flow
print(f"\n📬 Total messages in history: {len(history)}\n")
for i, msg in enumerate(history):
    role = msg["role"] if isinstance(msg, dict) else msg.role
    
    if role == "system":
        print(f"  [{i}] 🔧 SYSTEM: (system prompt)")
    elif role == "user":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        print(f"  [{i}] 👤 USER: {content[:80]}")
    elif role == "assistant":
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [{i}] 🤖 ASSISTANT: tool_call → {tc.function.name}({tc.function.arguments})")
        else:
            content = msg["content"] if isinstance(msg, dict) else msg.content
            print(f"  [{i}] 🤖 ASSISTANT: {content[:80]}...")
    elif role == "tool":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        print(f"  [{i}] 🔧 TOOL RESULT: {content[:80]}")

### 💡 Key Insight: The Message History IS the Agent's Memory

Look at the message flow above. Every message stays in the history:

```
[0] system    →  "You are a helpful assistant..."
[1] user      →  "What's the weather in London and what is 32 * 9?"
[2] assistant →  tool_call: get_weather({city: "London"})
[3] tool      →  "Weather in London: 8°C, rainy, humidity 85%"
[4] assistant →  tool_call: calculate({expression: "32 * 9"})
[5] tool      →  "Result: 32 * 9 = 288"
[6] assistant →  "The weather in London is 8°C and rainy. 32 × 9 = 288."
```

The LLM sees **all previous messages** on every call. That's how it knows:
- What the user asked
- What tools it already called
- What results it got
- What's left to do

**This is the agent's "short-term memory"** — it only lasts for one request.

---

# 🏗️ STAGE 2: Rebuild with LangGraph

In Stage 1, you built ~80 lines of code to make an agent work. That code handled:
- The ReAct loop
- Message history management
- Tool call parsing
- Tool execution
- Max iteration safety

But every agent needs this same code. That's why **frameworks** exist — they give you
all of this **out of the box** so you can focus on your tools and prompts.

### What is LangGraph?

**LangGraph** is a framework by LangChain that models agents as **graphs**:
- **Nodes** = functions that do something (call LLM, execute tools, process data)
- **Edges** = transitions between nodes (what happens next)
- **State** = shared data that flows through the graph (messages, results, etc.)

The `create_react_agent()` function creates a complete ReAct agent graph for you:

```
┌─────────────────────────────────────────────────────────────┐
│                  LangGraph Agent                            │
│                                                             │
│   ▶ START ──▶ [Agent Node] ──▶ Should I use a tool?        │
│                    ▲               │          │             │
│                    │          Yes  │     No   │             │
│                    │               ▼          ▼             │
│                    └── [Tool Node]      ⏹ END              │
│                                                             │
│   This is the SAME ReAct loop from Stage 1 —               │
│   but the framework handles it for you!                    │
└─────────────────────────────────────────────────────────────┘
```

### What changes from Stage 1?

| What You Did in Stage 1 | What LangGraph Does For You |
|-------------------------|---------------------------|
| Wrote the while loop | `create_react_agent()` handles the loop |
| Parsed `tool_calls` from JSON | `@tool` decorator auto-generates schemas |
| Managed `messages` list manually | State managed automatically |
| Added `max_iterations` | Built-in with configurable recursion limit |
| Wrote error handling | Built-in error handling and retries |

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ──────────────────────────────────────────────────────
# Step 2.1: Define tools using LangChain's @tool decorator
#
# Compare this to Stage 1 — NO JSON schema needed!
# The decorator reads the function name, docstring, and 
# type hints to auto-generate the tool schema.
#
# The docstring IS the tool description the LLM reads.
# ──────────────────────────────────────────────────────

@tool
def get_weather_lc(city: str) -> str:
    """Get the current weather for a given city.
    Use this when the user asks about weather or temperature."""
    return get_weather(city)  # Reuse our function from Stage 1

@tool
def calculate_lc(expression: str) -> str:
    """Evaluate a mathematical expression.
    Use this for any math calculations."""
    return calculate(expression)  # Reuse our function from Stage 1

@tool
def search_knowledge_lc(query: str) -> str:
    """Search the company knowledge base for policies and information.
    Use this when the user asks about company rules or guidelines."""
    return search_knowledge(query)  # Reuse our function from Stage 1

# ──────────────────────────────────────────────────────
# Step 2.2: Create the agent
#
# THIS IS THE ENTIRE AGENT — one function call!
# Compare this to the ~80 lines we wrote in Stage 1.
#
# create_react_agent() creates a complete graph with:
#   - An "agent" node (calls the LLM)
#   - A "tools" node (executes tools)
#   - Edges between them (the ReAct loop)
#   - Automatic stopping when the LLM gives a final answer
# ──────────────────────────────────────────────────────

# Connect LangChain to Azure OpenAI
llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
)
tools = [get_weather_lc, calculate_lc, search_knowledge_lc]

# One line to create a complete ReAct agent:
agent = create_react_agent(llm, tools)

print("✅ LangGraph agent created!")
print(f"   Model: {os.getenv('AZURE_OPENAI_DEPLOYMENT_GPT41', 'gpt-41')}")
print(f"   Tools: {[t.name for t in tools]}")
print()
print("💡 This agent does EXACTLY what our Stage 1 agent did —")
print("   but in ~15 lines instead of ~80!")

In [ ]:
# ──────────────────────────────────────────────────────
# Step 2.3: Run the LangGraph agent
#
# Let's run the SAME questions from Stage 1 — you'll get 
# the same results, but notice how much simpler the code is.
#
# agent.invoke() does everything our run_agent() function did:
#   - Sends messages to the LLM
#   - Detects tool_calls in the response
#   - Executes the tools
#   - Sends results back
#   - Loops until done
# ──────────────────────────────────────────────────────

result = agent.invoke(
    {"messages": [("user", "What's the weather in Tel Aviv?")]}
)

# Print all messages to see the flow
print("📬 LangGraph message flow:\n")
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  🤖 {role}: tool_call → {tc['name']}({tc['args']})")
    elif hasattr(msg, 'content') and msg.content:
        prefix = "👤" if "Human" in role else "🤖" if "AI" in role else "🔧"
        print(f"  {prefix} {role}: {msg.content[:100]}")

In [ ]:
# ──────────────────────────────────────────────────────
# Same multi-tool question from Stage 1 Test 3
#
# The agent handles: weather lookup + knowledge search + 
# calculation — all automatically, same as our raw code.
# ──────────────────────────────────────────────────────
result = agent.invoke({
    "messages": [(
        "user", 
        "I'm planning a trip to Tokyo. What's the weather there? "
        "Also, what's the daily meal expense limit? "
        "If I eat 3 meals at $15 each, am I within budget?"
    )]
})

# Just print the final answer
final = result["messages"][-1].content
print(f"📤 Final answer:\n\n{final}")

### 🤔 Stage 1 vs Stage 2: What Changed?

| Aspect | Stage 1 (Raw Python) | Stage 2 (LangGraph) |
|--------|---------------------|----------------------|
| **Lines of code** | ~80 | ~15 |
| **ReAct loop** | We wrote it manually | Framework handles it |
| **Tool schemas** | JSON by hand | `@tool` decorator auto-generates |
| **Message management** | We managed the list | Framework manages state |
| **Error handling** | We had to add it | Built-in |
| **Max iterations** | We implemented it | Built-in |

**The framework does exactly what we built by hand** — but it's tested, handles edge cases, and is production-ready.

---

# 🏗️ STAGE 3: Add Memory and Persistence

Our agent from Stage 2 has a big problem: **it forgets everything between requests.**

This is like talking to someone with amnesia — every conversation starts from zero.
The agent doesn't remember your name, your preferences, or what you discussed 5 minutes ago.

### Why does this happen?

Remember the message history from Stage 1? Each `agent.invoke()` call starts with a
**fresh, empty** message list. The previous conversation is gone.

### How to fix it?

LangGraph provides **checkpointing** — after each interaction, the agent's state
(all messages, tool results, everything) is automatically saved to storage.

When the same `thread_id` is used again, LangGraph loads the previous state
and continues the conversation.

Think of it like **WhatsApp threads** — each thread remembers all previous messages.

### Let's see the problem first:

The cell below demonstrates that our current agent has NO memory:

In [ ]:
# ──────────────────────────────────────────────────────
# DEMONSTRATION: The agent has NO memory
#
# We send two separate messages. The second one asks
# "what's my name?" — but the agent has no idea,
# because each invoke() starts with a fresh state.
# ──────────────────────────────────────────────────────

result1 = agent.invoke({"messages": [("user", "My name is Roi.")]})
print("Response 1:", result1["messages"][-1].content)
print()

result2 = agent.invoke({"messages": [("user", "What's my name?")]})
print("Response 2:", result2["messages"][-1].content)
print()
print("😔 It doesn't remember! Each invoke() starts with a fresh, empty state.")

### The Fix: Thread-Based Memory with Checkpointing

LangGraph solves this with a **checkpointer** — a storage backend that saves the
agent's state after each interaction.

Here's how it works:
1. You create an agent with a `checkpointer` (saves state)
2. When you call `invoke()`, you pass a `thread_id` (identifies the conversation)
3. LangGraph automatically loads previous state for that thread
4. After the call, LangGraph saves the updated state

```
Call 1: thread_id="user-123" → saves: [system, user: "hi", assistant: "hello"]
Call 2: thread_id="user-123" → loads previous state, continues conversation
Call 3: thread_id="different-456" → separate conversation, separate state
```

> 💡 **Multi-tenancy!** Different `thread_id` = different conversation = complete isolation.
> This is how production agents handle multiple users simultaneously.

In [ ]:
# ──────────────────────────────────────────────────────
# Create agent WITH memory (checkpointing)
#
# MemorySaver() stores state in-memory (RAM).
# For production, you'd use SqliteSaver or PostgresSaver
# so state survives server restarts.
#
# The thread_id is like a WhatsApp chat — each thread
# has its own conversation history, completely isolated.
# ──────────────────────────────────────────────────────

from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()  # In-memory storage (for labs only)
agent_with_memory = create_react_agent(llm, tools, checkpointer=memory)

# thread_id identifies THIS conversation
config = {"configurable": {"thread_id": "user-roi-123"}}

# First message — tell the agent who we are
result1 = agent_with_memory.invoke(
    {"messages": [("user", "My name is Roi. I work in the analytics team.")]},
    config=config
)
print("Response 1:", result1["messages"][-1].content)
print()

# Second message — SAME thread_id → the agent remembers!
result2 = agent_with_memory.invoke(
    {"messages": [("user", "What's my name and which team do I work in?")]},
    config=config
)
print("Response 2:", result2["messages"][-1].content)
print()
print("🎉 It remembers! The thread_id connects the conversations.")

In [ ]:
# Different thread_id = different conversation = no memory
config_new = {"configurable": {"thread_id": "different-user-456"}}

result3 = agent_with_memory.invoke(
    {"messages": [("user", "What's my name?")]},
    config=config_new
)
print("Different thread:", result3["messages"][-1].content)
print("\n🔒 Different thread = isolated conversation. This is multi-tenancy!")

---

# 🎓 Summary: What We Built and Learned

### The Three Stages

```
Stage 1                   Stage 2                   Stage 3
Raw Python                LangGraph                 + Memory
━━━━━━━━━━                ━━━━━━━━━                ━━━━━━━━
                                                    
✅ Built ReAct loop       ✅ Same agent,            ✅ Agent remembers
   from scratch              15 lines                  conversations
✅ Managed messages       ✅ Framework handles      ✅ Thread isolation
   manually                  everything                (multi-tenant)
✅ Parsed tool calls      ✅ @tool decorator        ✅ Checkpointing
   from JSON                 auto-generates            saves state
                                                    
~80 lines of code         ~15 lines of code         ~25 lines of code
```

### Key Concepts Learned

| Concept | What It Means |
|---------|---------------|
| **Agent** | LLM + Tools + Loop |
| **ReAct** | Think → Act → Observe → repeat |
| **Tool Calling** | LLM outputs JSON, YOUR code executes the tool |
| **Message History** | The agent's short-term memory within a request |
| **Max Iterations** | Safety limit to prevent infinite loops |
| **LangGraph** | Framework that models agents as graphs (nodes + edges) |
| **Checkpointing** | Saving agent state for memory across requests |
| **Thread ID** | Identifier that separates conversations (multi-tenancy) |

### What's Next?

In **Lab 02**, we'll add **smart model routing** — using cheap models for simple tasks and expensive models for complex ones. You'll see how this can cut costs by 80%.

---

> **[← Back to Labs Overview](../README.md)** | **[→ Lab 02: Smart Model Routing](../lab-02-model-routing/README.md)**